# EDA — line-mask dataset

Quick exploratory analysis of the (image, mask) pairs collected from the Unity sim.
Goal: validate dataset quality and surface design constraints for the U-Net.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

from src.config import load_config

cfg = load_config()
manifest = pd.read_csv(cfg.paths.data_raw / 'manifest.csv')
print(f'{len(manifest)} pairs in dataset')
manifest.head()

## 1. Source breakdown

In [ ]:
manifest['source'].value_counts().plot.bar(title='Pairs per source folder')

## 2. Sample pairs (image / binarized mask / overlay)

In [ ]:
n_show = 4
sampled = manifest.sample(n_show, random_state=42)
fig, axes = plt.subplots(n_show, 3, figsize=(12, 3 * n_show))
for ax_row, (_, row) in zip(axes, sampled.iterrows()):
    img = np.array(Image.open(cfg.paths.data_raw / 'images' / f"{row['pair_id']}.png"))
    mask_rgb = np.array(Image.open(cfg.paths.data_raw / 'masks' / f"{row['pair_id']}.png"))
    mask_bin = mask_rgb.min(axis=-1) >= cfg.mask_threshold
    overlay = img.copy()
    overlay[mask_bin] = [255, 0, 0]
    ax_row[0].imshow(img); ax_row[0].set_title(f"image {row['pair_id']}"); ax_row[0].axis('off')
    ax_row[1].imshow(mask_bin, cmap='gray'); ax_row[1].set_title('binary mask'); ax_row[1].axis('off')
    ax_row[2].imshow(overlay); ax_row[2].set_title('overlay'); ax_row[2].axis('off')
plt.tight_layout()

## 3. Foreground ratio (class imbalance check)

In [ ]:
ratios = []
for pair_id in manifest['pair_id']:
    m = np.array(Image.open(cfg.paths.data_raw / 'masks' / f'{pair_id}.png'))
    ratios.append((m.min(axis=-1) >= cfg.mask_threshold).mean())
ratios = np.array(ratios)
print(f'Foreground (line) pixel ratio — mean: {ratios.mean():.4f}, median: {np.median(ratios):.4f}, range: [{ratios.min():.4f}, {ratios.max():.4f}]')
plt.hist(ratios, bins=30); plt.xlabel('foreground ratio'); plt.ylabel('count'); plt.title('Class balance per sample')

## 4. Image brightness distribution
Helps decide augmentation strength: if all images are dark/bright, less brightness aug needed.

In [ ]:
means = []
for pair_id in manifest['pair_id']:
    img = np.array(Image.open(cfg.paths.data_raw / 'images' / f'{pair_id}.png'))
    means.append(img.mean())
means = np.array(means)
plt.hist(means, bins=30); plt.xlabel('mean RGB intensity'); plt.ylabel('count'); plt.title('Image brightness distribution')
print(f'Mean brightness — mean: {means.mean():.1f}, std: {means.std():.1f}')

## 5. Mask spatial distribution heatmap
Where are the lines on average? Helps validate the raycast origin choice (bottom-center).

In [ ]:
heatmap = None
for pair_id in manifest['pair_id']:
    m = np.array(Image.open(cfg.paths.data_raw / 'masks' / f'{pair_id}.png'))
    bin_m = (m.min(axis=-1) >= cfg.mask_threshold).astype(np.float32)
    heatmap = bin_m if heatmap is None else heatmap + bin_m
heatmap /= len(manifest)
plt.figure(figsize=(8, 5))
plt.imshow(heatmap, cmap='hot'); plt.colorbar(label='line probability')
plt.title('Average mask — bright pixels = lines often appear there')
plt.scatter([heatmap.shape[1] // 2], [heatmap.shape[0] - 1], c='cyan', s=50, marker='x', label='raycast origin')
plt.legend()